# Código corregido v1 · Tesis: volatilidad de precios internacionales y saldo comercial de Guatemala (2010–2024)

José Andrés Estrada Morales · Carné 1031722

Parte de `modelo_tesis_corregido.py` (v0) y aplica las correcciones de resultados del checklist final.
Cada bloque indica entre corchetes el ítem que atiende: [C19], [C23], [C25], [C28]–[C31], [C33]–[C36],
[C38]–[C46]. El archivo `README_resultados.md` que genera la ejecución relaciona cada salida con su ítem.

Decisiones de datos (no son correcciones del código):
- Comercio: base corregida (perfil mensual FMI-DOT ajustado a los totales anuales del Banco de Guatemala).
- Cardamomo: se conservan las 23 interpolaciones lineales tal como están en la base (decisión del autor; C24 no aplica).

Uso: deja este archivo y Base_datos_tesis.xlsx en la misma carpeta y ejecuta todo. Desde terminal:
python codigo_tesis_v1.py --dependiente saldo_norm   (o --dependiente balanza --salida resultados_v1_saldo_USD)
Dependencias probadas: arch 7.2.0, statsmodels 0.14.5, pandas 2.3.1, numpy 2.0.2, scipy, matplotlib, openpyxl.

In [ ]:
from pathlib import Path
import argparse
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
if 'ipykernel' not in sys.modules:  # en terminal se guardan las figuras sin abrir ventanas
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm
import arch
import statsmodels
from arch import arch_model
from arch.unitroot import ADF, PhillipsPerron, KPSS, ZivotAndrews
from statsmodels.tsa.api import VAR
from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey, acorr_ljungbox, het_arch,
                                          het_breuschpagan, recursive_olsresiduals)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

MAPA = {
    'Periodo': 'fecha',
    'Exportaciones FOB (US$ millones)': 'exportaciones',
    'Importaciones CIF (US$ millones)': 'importaciones',
    'Balanza comercial (US$ millones)': 'balanza',
    'Café Arábica (US$/kg)': 'cafe',
    'Azúcar mundial (US$/kg)': 'azucar',
    'Cardamomo: valor unitario exportación (US$/kg)': 'cardamomo',
    'Banano EE.UU. (US$/kg)': 'banano',
    'Petróleo WTI (US$/bbl)': 'petroleo',
    'Tipo de cambio promedio (Q/USD)': 'tipo_cambio',
    'Remesas familiares (US$ millones)': 'remesas',
    'ITCER global (índice)': 'reer',
}
PRECIOS = ['cafe', 'azucar', 'cardamomo', 'banano', 'petroleo']
NOMBRES = dict(cafe='Café', azucar='Azúcar', cardamomo='Cardamomo', banano='Banano', petroleo='Petróleo WTI',
               balanza='Saldo comercial', saldo_norm='Saldo normalizado', reer='ITCER',
               tipo_cambio='Tipo de cambio nominal', remesas='Remesas')
CONTROLES = ['reer', 'tipo_cambio', 'remesas']
MAX_P = 12          # Máximo rezago del saldo evaluado [C38]
MAX_Q = 3           # Máximo rezago de cada regresor evaluado [C38]
HB_GARCH = 3        # Muestra común para comparar ecuaciones de media AR(0)–AR(3) [C28]
SEMILLA = 20261108
FUENTE_COMERCIO = 'FMI-DOT y Banco de Guatemala'
FUENTE_PRECIOS = 'Banco Mundial (Pink Sheet) y UN Comtrade (cardamomo)'
ARCHIVO = 'Base_datos_tesis.xlsx'   # base en la misma carpeta que este archivo
SALIDA = 'resultados_v1_saldo_normalizado'


def formato_p(p):
    return 'p < 0.0001' if p < 1e-4 else f'p = {p:.4f}'

In [ ]:
def cargar_datos(archivo):
    raw = pd.read_excel(archivo, sheet_name='BASE_CORREGIDA')
    faltan = set(MAPA) - set(raw.columns)
    if faltan:
        raise ValueError(f'Encabezados faltantes: {sorted(faltan)}')
    df = raw[list(MAPA)].rename(columns=MAPA)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.set_index('fecha').sort_index()
    if not df.index.equals(pd.date_range('2010-01-01', '2024-12-01', freq='MS')):
        raise ValueError('Se requieren los 180 meses consecutivos de 2010-01 a 2024-12.')
    df = df.apply(pd.to_numeric, errors='raise')
    if not np.isfinite(df.to_numpy()).all() or (df[PRECIOS] <= 0).any().any():
        raise ValueError('Datos ausentes, no finitos o precios no positivos.')
    np.testing.assert_allclose(df.balanza, df.exportaciones - df.importaciones, atol=1e-6)
    calidad = pd.read_excel(archivo, sheet_name='CALIDAD_CARDAMOMO')
    calidad['Periodo'] = pd.to_datetime(calidad['Periodo'])
    bandera = calidad.set_index('Periodo').reindex(df.index)['Indicador imputado'].astype(int)
    # [C19] Saldo normalizado: saldo / (exportaciones + importaciones) × 100.
    df['saldo_norm'] = 100 * df.balanza / (df.exportaciones + df.importaciones)
    df['covid'] = ((df.index >= '2020-03-01') & (df.index <= '2021-12-01')).astype(float)
    return df, bandera


def diagnostico_precios(df, salida):
    """[C25] Valores repetidos (retorno cero) y retornos extremos por serie, para verificar contra la fuente."""
    filas, extremos = [], []
    for c in PRECIOS:
        r = 100 * np.log(df[c]).diff().dropna()
        repetidos = r.index[r.abs() < 1e-10]
        rachas, actual = [], 0
        for v in (r.abs() < 1e-10):
            actual = actual + 1 if v else 0
            rachas.append(actual)
        filas.append(dict(serie=c, meses_con_valor_repetido=len(repetidos),
                          racha_mas_larga=int(max(rachas)) + (1 if max(rachas) else 0),
                          fechas_repetidas=', '.join(d.strftime('%Y-%m') for d in repetidos)))
        for fecha, val in r.abs().sort_values(ascending=False).head(5).items():
            extremos.append(dict(serie=c, fecha=fecha.strftime('%Y-%m'), retorno=float(r[fecha]),
                                 precio_anterior=float(df[c].shift()[fecha]), precio=float(df[c][fecha])))
    pd.DataFrame(filas).to_csv(salida / '00_diagnostico_precios_repetidos.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(extremos).to_csv(salida / '00_diagnostico_retornos_extremos.csv', index=False, encoding='utf-8-sig')
    return pd.DataFrame(filas)


def diccionario(archivo, salida):
    """[C27] Diccionario de variables a partir de la hoja FUENTES_METODO de la base."""
    f = pd.read_excel(archivo, sheet_name='FUENTES_METODO', header=None)
    inicio = f.index[f.iloc[:, 0].astype(str).str.strip() == 'Variable'][0]
    fin = f.index[f.iloc[:, 0].astype(str).str.strip() == 'Problema detectado'][0]
    tabla = f.iloc[inicio + 1:fin].dropna(how='all')
    tabla.columns = f.iloc[inicio].tolist()
    tabla = tabla.dropna(subset=['Variable'])
    derivadas = pd.DataFrame([
        dict(Variable='Saldo normalizado', **{'Serie exacta': 'Saldo / (exportaciones + importaciones) × 100',
             'Frecuencia': 'Mensual', 'Unidad final': '%', 'Transformación': 'Cálculo propio [C19]',
             'Fuente': FUENTE_COMERCIO, 'URL': ''}),
        dict(Variable='Retorno de precio', **{'Serie exacta': '100 × Δln(precio)', 'Frecuencia': 'Mensual',
             'Unidad final': '% mensual', 'Transformación': 'Cálculo propio', 'Fuente': FUENTE_PRECIOS, 'URL': ''}),
        dict(Variable='Volatilidad condicional', **{'Serie exacta': 'Desviación estándar condicional σt del modelo seleccionado',
             'Frecuencia': 'Mensual', 'Unidad final': 'puntos porcentuales de retorno mensual (no anualizada)',
             'Transformación': 'GARCH/GJR/EGARCH seleccionado por BIC con diagnósticos [C28]', 'Fuente': 'Cálculo propio', 'URL': ''}),
        dict(Variable='Volatilidad móvil', **{'Serie exacta': 'Desviación estándar de los retornos en ventana de 12 meses',
             'Frecuencia': 'Mensual', 'Unidad final': 'puntos porcentuales de retorno mensual',
             'Transformación': 'Medida no GARCH de robustez [C31]', 'Fuente': 'Cálculo propio', 'URL': ''}),
        dict(Variable='Dummies estacionales', **{'Serie exacta': '11 indicadores mensuales centrados (febrero–diciembre)',
             'Frecuencia': 'Mensual', 'Unidad final': '—', 'Transformación': 'Regresores fijos del ARDL y del VAR [C39]',
             'Fuente': 'Construcción propia', 'URL': ''}),
    ])
    out = pd.concat([tabla, derivadas], ignore_index=True)
    out.to_csv(salida / '00_diccionario_variables.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Retorno: r_t = 100 × [ln(P_t) − ln(P_{t−1})]. La volatilidad es σ_t, la desviación estándar condicional
# del retorno mensual, en puntos porcentuales y SIN anualizar [C32].
# Candidatos iguales para las cinco series, con la misma muestra (hold_back = 3) [C30]:
#   media AR(0)–AR(3) [C29] × varianza GARCH(1,1), GJR-GARCH(1,1,1), EGARCH(1,1,1) × innovaciones normal, t, t asimétrica, GED.
# Un candidato es admisible si: converge; persistencia < 0.999; el parámetro de forma no está en la frontera
# (t y t asimétrica ν ≥ 2.5; GED ν ≥ 1.05); Ljung–Box(12) de residuos estandarizados y de sus cuadrados con p > 0.05;
# ARCH-LM(12) posterior con p > 0.05. Entre los admisibles se elige el menor BIC [C28].
FAMILIAS = {'GARCH': dict(vol='GARCH', p=1, o=0, q=1),
            'GJR-GARCH': dict(vol='GARCH', p=1, o=1, q=1),
            'EGARCH': dict(vol='EGARCH', p=1, o=1, q=1)}
DISTRIBUCIONES = ['normal', 't', 'skewt', 'ged']


def _arch(s, ar, familia, dist, hold_back=HB_GARCH):
    kw = dict(mean='AR' if ar else 'Constant', dist=dist, rescale=False, hold_back=hold_back)
    if ar:
        kw['lags'] = ar
    if familia == 'Constante':
        return arch_model(s, vol='Constant', **kw)
    return arch_model(s, **FAMILIAS[familia], **kw)


def _persistencia(familia, p):
    if familia == 'GARCH':
        return p['alpha[1]'] + p['beta[1]']
    if familia == 'GJR-GARCH':
        return p['alpha[1]'] + p['beta[1]'] + p['gamma[1]'] / 2
    if familia == 'EGARCH':
        return abs(p['beta[1]'])
    return np.nan


def _forma(dist, p):
    return {'t': p.get('nu', np.nan), 'skewt': p.get('eta', np.nan), 'ged': p.get('nu', np.nan)}.get(dist, np.nan)


def _frontera(dist, forma):
    return (dist in ('t', 'skewt') and forma < 2.5) or (dist == 'ged' and forma < 1.05)


def _diagnosticos(r, ar):
    z = r.std_resid.dropna()
    return dict(LB12_residuos_p=float(acorr_ljungbox(z, lags=[12], model_df=ar)['lb_pvalue'].iloc[0]),
                LB12_cuadrados_p=float(acorr_ljungbox(z**2, lags=[12])['lb_pvalue'].iloc[0]),
                ARCH_LM12_post_p=float(het_arch(z, nlags=12)[1]))


def estimar_volatilidad(s, serie):
    filas, ajustes = [], {}
    for ar in range(0, 4):
        for familia in list(FAMILIAS) + ['Constante']:
            for dist in DISTRIBUCIONES:
                try:
                    r = _arch(s, ar, familia, dist).fit(disp='off', options={'maxiter': 5000})
                except Exception as exc:  # se registra y se descarta
                    filas.append(dict(serie=serie, AR=ar, varianza=familia, distribucion=dist, error=str(exc)))
                    continue
                p = r.params
                fila = dict(serie=serie, AR=ar, varianza=familia, distribucion=dist,
                            convergencia=int(r.convergence_flag), logverosimilitud=float(r.loglikelihood),
                            parametros=len(p), n=int(r.nobs), AIC=float(r.aic), BIC=float(r.bic),
                            persistencia=float(_persistencia(familia, p)), forma=float(_forma(dist, p)),
                            **_diagnosticos(r, ar))
                fila['forma_en_frontera'] = bool(_frontera(dist, fila['forma']))
                fila['admisible'] = bool(familia != 'Constante' and fila['convergencia'] == 0
                                         and np.isfinite(r.params).all() and fila['persistencia'] < 0.999
                                         and not fila['forma_en_frontera'] and fila['LB12_residuos_p'] > .05
                                         and fila['LB12_cuadrados_p'] > .05 and fila['ARCH_LM12_post_p'] > .05)
                filas.append(fila)
                ajustes[(ar, familia, dist)] = r
    tabla = pd.DataFrame(filas)
    if tabla.n.dropna().nunique() != 1:
        raise AssertionError(f'{serie}: los candidatos no usan la misma muestra.')
    adm = tabla[tabla.admisible == True]
    if adm.empty:  # Respaldo documentado: GARCH(1,1) AR(1) normal por cuasi-máxima verosimilitud.
        clave, motivo = (1, 'GARCH', 'normal'), 'Ningún candidato admisible: respaldo GARCH(1,1) QML'
    else:
        g = adm.loc[adm.BIC.idxmin()]
        clave, motivo = (int(g.AR), g.varianza, g.distribucion), 'Menor BIC entre candidatos admisibles'
    r = ajustes[clave]
    ar, familia, dist = clave
    const = ajustes[(ar, 'Constante', dist)]
    lr = 2 * (r.loglikelihood - const.loglikelihood)
    gl = len(r.params) - len(const.params)
    media = OLS(s.iloc[HB_GARCH:], add_constant(pd.concat({f'L{j}': s.shift(j) for j in range(1, ar + 1)}, axis=1).iloc[HB_GARCH:])
                ).fit() if ar else OLS(s.iloc[HB_GARCH:], np.ones(len(s) - HB_GARCH)).fit()
    p = r.params
    info = dict(serie=serie, ecuacion_media=f'AR({ar})' if ar else 'Constante', varianza=familia,
                distribucion=dist, criterio=motivo, n=int(r.nobs), AIC=float(r.aic), BIC=float(r.bic),
                ARCH_LM12_previo_p=float(het_arch(media.resid, nlags=12)[1]),
                LR_vs_varianza_constante=float(lr), LR_gl=int(gl), LR_p=float(chi2.sf(lr, gl)),
                BIC_varianza_constante=float(const.bic),
                persistencia=float(_persistencia(familia, p)), forma=float(_forma(dist, p)),
                omega=float(p['omega']), positividad=bool(p['omega'] > 0 and (familia == 'EGARCH' or
                (p['alpha[1]'] >= 0 and p['beta[1]'] >= 0))),
                estacionaria_en_varianza=bool(_persistencia(familia, p) < 1),
                vol_media=float(r.conditional_volatility.mean()), vol_min=float(r.conditional_volatility.min()),
                vol_max=float(r.conditional_volatility.max()), **_diagnosticos(r, ar))
    info['heterocedasticidad_condicional'] = ('Sí (LR p < 0.05)' if info['LR_p'] < .05 else
                                              'No significativa (LR p ≥ 0.05): σt casi constante')
    parametros = pd.DataFrame({'serie': serie, 'parametro': p.index, 'coef': p.values,
                               'ee_robusto': r.std_err.values, 't': r.tvalues.values, 'p': r.pvalues.values})
    return r, info, tabla, parametros


def volatilidades(df, salida):
    sel, cand, pars, modelos, texto = [], [], [], {}, []
    for c in PRECIOS:
        s = df[f'r_{c}'].dropna()
        r, info, tabla, parametros = estimar_volatilidad(s, c)
        modelos[c] = r
        df[f'vol_{c}'] = r.conditional_volatility
        # [C30] Robustez de comparabilidad: la misma especificación para todas, GARCH(1,1) AR(1) normal (QML).
        u = _arch(s, 1, 'GARCH', 'normal').fit(disp='off', options={'maxiter': 5000})
        df[f'volg_{c}'] = u.conditional_volatility
        # [C31] Medida no GARCH: desviación estándar móvil de 12 meses de los retornos.
        df[f'volm_{c}'] = s.rolling(12).std()
        sel.append(info); cand.append(tabla); pars.append(parametros)
        texto.append(f'{"=" * 100}\n{NOMBRES[c].upper()} · {info["ecuacion_media"]} – {info["varianza"]} – '
                     f'{info["distribucion"]} · {info["criterio"]}\n{"=" * 100}\n{r.summary()}\n')
    pd.DataFrame(sel).to_csv(salida / '01_volatilidad_seleccion_y_diagnosticos.csv', index=False, encoding='utf-8-sig')
    pd.concat(cand).to_csv(salida / '01_volatilidad_candidatos.csv', index=False, encoding='utf-8-sig')
    pd.concat(pars).to_csv(salida / '01_volatilidad_parametros.csv', index=False, encoding='utf-8-sig')
    (salida / '01_anexo_10_2_salidas_GARCH.txt').write_text('\n'.join(texto), encoding='utf-8')  # [C34]
    return pd.DataFrame(sel), modelos

In [ ]:
def _ur(x, trend):
    x = x.dropna()
    out = {}
    for nombre, prueba in [('ADF', lambda: ADF(x, trend=trend, method='aic')),
                           ('PP', lambda: PhillipsPerron(x, trend=trend)),
                           ('KPSS', lambda: KPSS(x, trend=trend)),
                           ('ZA', lambda: ZivotAndrews(x, trend=trend, method='aic'))]:
        try:
            t = prueba()
            out[nombre] = float(t.pvalue)
            if nombre == 'ZA':
                out['ZA_estadistico'] = float(t.stat)
        except Exception as exc:  # p. ej. serie casi constante: se registra como no calculable
            out[nombre] = np.nan
            out[f'{nombre}_nota'] = type(exc).__name__
    return out


def _estacionaria(t):
    """ADF o PP rechazan raíz unitaria y KPSS no rechaza estacionariedad, con 'c' o con 'ct'."""
    return any(((t[d]['ADF'] < .05) or (t[d]['PP'] < .05)) and t[d]['KPSS'] > .05 for d in ('c', 'ct'))


def integracion(df, variables, salida):
    filas, resumen = [], []
    for v in variables:
        x = df[v].dropna()
        if x.std() < 0.01 * abs(x.mean()):  # σt sin variación relevante: las pruebas no son informativas
            resumen.append(dict(variable=v, nombre=v, orden='Casi constante',
                                regla='Desviación estándar < 1% de la media; no se clasifica'))
            continue
        pruebas = {}
        for orden, serie in [('nivel', x), ('diff', x.diff()), ('diff2', x.diff().diff())]:
            pruebas[orden] = {d: _ur(serie, d) for d in ('c', 'ct')}
            for d in ('c', 'ct'):
                filas.append(dict(variable=v, transformacion=orden, deterministicos=d, **pruebas[orden][d]))
        if any(np.isnan(pruebas['nivel'][d][t]) for d in ('c', 'ct') for t in ('ADF', 'PP', 'KPSS')):
            resumen.append(dict(variable=v, nombre=v, orden='No calculable',
                                regla='Alguna prueba no se pudo calcular (serie casi constante); ver detalle'))
            continue
        nivel = _estacionaria(pruebas['nivel'])
        conflicto = (not nivel and any(pruebas['nivel'][d]['ADF'] < .05 or pruebas['nivel'][d]['PP'] < .05
                                       for d in ('c', 'ct')))
        za_nivel = min(pruebas['nivel'][d]['ZA'] for d in ('c', 'ct')) < .05
        if nivel:
            orden, regla = 'I(0)', 'ADF/PP rechazan y KPSS no rechaza en niveles'
        elif conflicto and za_nivel:
            orden, regla = 'I(0) con quiebre', 'ADF/PP rechazan, KPSS rechaza; Zivot–Andrews rechaza con quiebre'
        elif all(pruebas['diff'][d]['ADF'] < .05 and pruebas['diff'][d]['PP'] < .05 for d in ('c', 'ct')):
            orden, regla = 'I(1)', 'No estacionaria en niveles; ADF y PP rechazan en primeras diferencias'
        elif _estacionaria(pruebas['diff']):
            orden, regla = 'I(1)', 'No estacionaria en niveles; primeras diferencias estacionarias'
        else:
            orden, regla = 'I(2) posible', 'Primeras diferencias no estacionarias'
        resumen.append(dict(variable=v, nombre=NOMBRES.get(v.replace('vol_', '').replace('volg_', '').replace('volm_', ''), v),
                            orden=orden, regla=regla,
                            ADF_c_nivel=pruebas['nivel']['c']['ADF'], PP_c_nivel=pruebas['nivel']['c']['PP'],
                            KPSS_c_nivel=pruebas['nivel']['c']['KPSS'], ZA_c_nivel=pruebas['nivel']['c']['ZA'],
                            ADF_ct_nivel=pruebas['nivel']['ct']['ADF'], PP_ct_nivel=pruebas['nivel']['ct']['PP'],
                            KPSS_ct_nivel=pruebas['nivel']['ct']['KPSS'], ZA_ct_nivel=pruebas['nivel']['ct']['ZA'],
                            ADF_c_diff=pruebas['diff']['c']['ADF'], PP_c_diff=pruebas['diff']['c']['PP'],
                            KPSS_c_diff=pruebas['diff']['c']['KPSS']))
    pd.DataFrame(filas).to_csv(salida / '02_integracion_detalle.csv', index=False, encoding='utf-8-sig')
    res = pd.DataFrame(resumen)
    res.to_csv(salida / '02_integracion_resumen.csv', index=False, encoding='utf-8-sig')
    return res

In [ ]:
def dummies_estacionales(index):
    """[C39] 11 dummies mensuales centradas: no alteran la media ni el término constante."""
    S = pd.get_dummies(index.month, prefix='m', dtype=float).iloc[:, 1:]
    S.index = index
    return S - S.mean()


def _ardl(y, X, p, q, fijo):
    return ARDL(y, lags=p, exog=X, order=q, trend='c', fixed=fijo, hold_back=MAX_P).fit()


def seleccionar_rezagos(y, X, fijo, criterio, nombre, salida):
    """[C38] Rejilla p = 1..12 con q común = 1..3; luego ajuste individual de cada q por coordenadas.
    La muestra es común a todos los candidatos (hold_back = 12). q ≥ 1 para conservar cada nivel en el UECM."""
    cols, rejilla = list(X.columns), []
    for p in range(1, MAX_P + 1):
        for q in range(1, MAX_Q + 1):
            r = _ardl(y, X, p, {c: q for c in cols}, fijo)
            rejilla.append(dict(p=p, q_comun=q, n=int(r.nobs), AIC=float(r.aic), BIC=float(r.bic)))
    rej = pd.DataFrame(rejilla)
    if rej.n.nunique() != 1:
        raise AssertionError('La rejilla de rezagos no usa una muestra común.')
    mejor = rej.loc[rej[criterio].idxmin()]
    p, q = int(mejor.p), {c: int(mejor.q_comun) for c in cols}
    valor = float(mejor[criterio])
    for _ in range(3):
        cambio = False
        for c in cols:
            for qq in range(1, MAX_Q + 1):
                prueba = dict(q, **{c: qq})
                v = getattr(_ardl(y, X, p, prueba, fijo), criterio.lower())
                if v < valor - 1e-9:
                    valor, q, cambio = v, prueba, True
        for pp in range(1, MAX_P + 1):
            v = getattr(_ardl(y, X, pp, q, fijo), criterio.lower())
            if v < valor - 1e-9:
                valor, p, cambio = v, pp, True
        if not cambio:
            break
    rej.to_csv(salida / f'03_rezagos_{nombre}_{criterio}.csv', index=False, encoding='utf-8-sig')
    return p, q, rej


def disenio_uecm(y, X, p, q, fijo):
    """Diseño explícito del UECM, con los mismos nombres que statsmodels."""
    D = pd.DataFrame({'const': 1.0, f'{y.name}.L1': y.shift(1)}, index=y.index)
    for c in X:
        D[f'{c}.L1'] = X[c].shift(1)
    for j in range(1, p):
        D[f'D.{y.name}.L{j}'] = y.diff().shift(j)
    for c in X:
        for j in range(0, q[c]):
            D[f'D.{c}.L{j}'] = X[c].diff().shift(j)
    if fijo is not None:
        D = D.join(fijo)
    return D.iloc[MAX_P:], y.diff().iloc[MAX_P:]


def _f(res, cols):
    R = np.zeros((len(cols), len(res.params)))
    for i, c in enumerate(cols):
        R[i, res.params.index.get_loc(c)] = 1
    t = res.f_test(R)
    return float(np.squeeze(t.fvalue)), float(t.pvalue)

In [ ]:
def valores_criticos(D, y_name, xcols, lam, nsim, seed):
    """Simulación con el diseño de la muestra (n, rezagos, dummies) bajo H0 de ausencia de relación en niveles.
    Límite inferior: regresores I(0); superior: regresores I(1). Para el contraste F de degeneración de los
    regresores (Fx) se usa además y estacionaria AR(1) con ρ = 1 + λ̂, y se toma el valor más exigente."""
    rng = np.random.default_rng(seed)
    n, k = len(D), len(xcols)
    p = sum(c.startswith(f'D.{y_name}.L') for c in D) + 1
    qs = {c: sum(col.startswith(f'D.{c}.L') for col in D) for c in xcols}
    fijos = [c for c in D if c.startswith('m_') or c in ('covid', 'quiebre')]
    extra = MAX_P + 50
    out = {}
    for escenario in ('I0', 'I1', 'y_estacionaria'):
        F, T, FX = [], [], []
        for _ in range(nsim):
            e = rng.standard_normal((n + extra, k + 1))
            if escenario == 'y_estacionaria':
                yy = np.zeros(n + extra)
                rho = min(max(1 + lam, -0.95), 0.95)
                for t in range(1, n + extra):
                    yy[t] = rho * yy[t - 1] + e[t, 0]
                xx = np.cumsum(e[:, 1:], axis=0)
            else:
                yy = np.cumsum(e[:, 0])
                xx = e[:, 1:] if escenario == 'I0' else np.cumsum(e[:, 1:], axis=0)
            yy, xx = yy[-(n + MAX_P + 1):], xx[-(n + MAX_P + 1):]
            dy = np.diff(yy)[-n:]
            cols = [np.ones(n), yy[-n - 1:-1]] + [xx[-n - 1:-1, j] for j in range(k)]
            dyy = np.diff(yy)
            for j in range(1, p):
                cols.append(dyy[-n - j:len(dyy) - j])
            dxx = np.diff(xx, axis=0)
            for j, c in enumerate(xcols):
                for l in range(qs[c]):
                    cols.append(dxx[len(dxx) - n - l:len(dxx) - l, j])
            Z = np.column_stack(cols + [D[c].to_numpy() for c in fijos])
            b, *_ = np.linalg.lstsq(Z, dy, rcond=None)
            u = dy - Z @ b
            s2 = u @ u / (n - Z.shape[1])
            V = s2 * np.linalg.inv(Z.T @ Z)
            idx = list(range(1, 2 + k))
            bb, VV = b[idx], V[np.ix_(idx, idx)]
            F.append(bb @ np.linalg.solve(VV, bb) / len(idx))
            T.append(b[1] / np.sqrt(V[1, 1]))
            ix = list(range(2, 2 + k))
            FX.append(b[ix] @ np.linalg.solve(V[np.ix_(ix, ix)], b[ix]) / k)
        out[escenario] = dict(F=np.array(F), t=np.array(T), Fx=np.array(FX))
    cv = {}
    for nivel in (90, 95, 99):
        cv[nivel] = dict(F_I0=np.percentile(out['I0']['F'], nivel), F_I1=np.percentile(out['I1']['F'], nivel),
                         t_I0=np.percentile(out['I0']['t'], 100 - nivel), t_I1=np.percentile(out['I1']['t'], 100 - nivel),
                         Fx_I0=max(np.percentile(out['I0']['Fx'], nivel), np.percentile(out['y_estacionaria']['Fx'], nivel)),
                         Fx_I1=max(np.percentile(out['I1']['Fx'], nivel), np.percentile(out['y_estacionaria']['Fx'], nivel)))
    return cv


def _decision(stat, inf, sup, cola='derecha'):
    if cola == 'izquierda':
        return 'Rechaza H0' if stat < sup else ('No rechaza H0' if stat > inf else 'Inconcluso')
    return 'Rechaza H0' if stat > sup else ('No rechaza H0' if stat < inf else 'Inconcluso')


def bounds(u_ols, u_sm, y_name, xcols, D, nsim, seed):
    niveles = [f'{y_name}.L1'] + [f'{c}.L1' for c in xcols]
    F, _ = _f(u_ols, niveles)
    t = float(u_ols.tvalues[f'{y_name}.L1'])
    Fx, _ = _f(u_ols, [f'{c}.L1' for c in xcols])
    cv = valores_criticos(D, y_name, xcols, float(u_ols.params[f'{y_name}.L1']), nsim, seed)
    c5 = cv[95]
    asin = None
    try:  # referencia asintótica de statsmodels (PSS, caso 3), solo para comparar
        bt = u_sm.bounds_test(case=3, asymptotic=True)
        tabla = bt.crit_vals
        asin = (float(tabla.loc[95., 'lower']), float(tabla.loc[95., 'upper']))
    except Exception:
        pass
    res = dict(F=F, F_I0_5=c5['F_I0'], F_I1_5=c5['F_I1'], F_decision=_decision(F, c5['F_I0'], c5['F_I1']),
               t=t, t_I0_5=c5['t_I0'], t_I1_5=c5['t_I1'], t_decision=_decision(t, c5['t_I0'], c5['t_I1'], 'izquierda'),
               Fx=Fx, Fx_I0_5=c5['Fx_I0'], Fx_I1_5=c5['Fx_I1'], Fx_decision=_decision(Fx, c5['Fx_I0'], c5['Fx_I1']),
               F_asintotico_I0_5=asin[0] if asin else np.nan, F_asintotico_I1_5=asin[1] if asin else np.nan)
    tres = (res['F_decision'] == 'Rechaza H0' and res['t_decision'] == 'Rechaza H0' and res['Fx_decision'] == 'Rechaza H0')
    res['relacion_en_niveles'] = ('Respaldada por F, t y Fx' if tres else
                                  'No respaldada: al menos un contraste no rechaza o es inconcluso (posible caso degenerado)')
    return res, cv

In [ ]:
def hac_lags(n):
    return int(np.floor(4 * (n / 100) ** (2 / 9)))


def estimar(ad, y_name, xcols, fijo, criterio, nombre, salida, vols, nsim=2000, seed=SEMILLA,
            p=None, q=None, completo=False):
    y, X = ad[y_name], ad[xcols]
    if p is None:
        p, q, _ = seleccionar_rezagos(y, X, fijo, criterio, nombre, salida)
    a = _ardl(y, X, p, q, fijo)
    u_sm = UECM.from_ardl(a.model).fit()
    D, dy = disenio_uecm(y, X, p, q, fijo)
    D = D[u_sm.params.index]
    conv = OLS(dy, D).fit()
    np.testing.assert_allclose(conv.params, u_sm.params, rtol=1e-6, atol=1e-6)
    np.testing.assert_allclose(conv.resid, a.resid, rtol=1e-6, atol=1e-6)
    n = int(conv.nobs)
    hc3 = OLS(dy, D).fit(cov_type='HC3')
    hac = OLS(dy, D).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags(n)})
    r2_niv = 1 - np.sum(a.resid ** 2) / np.sum((y.iloc[MAX_P:] - y.iloc[MAX_P:].mean()) ** 2)
    r2_dif = float(conv.rsquared)
    k = len(conv.params)
    b, cv = bounds(conv, u_sm, y_name, xcols, D, nsim, seed)
    # [C44] Pruebas conjuntas de las cinco volatilidades: niveles, diferencias y ambas; tres covarianzas.
    vs = [v for v in vols if v in xcols]
    conj = {}
    for etiqueta, cols in [('diferencias', [c for c in D if any(c.startswith(f'D.{v}.L') for v in vs)]),
                           ('niveles', [f'{v}.L1' for v in vs]),
                           ('todas', [c for c in D if any(c.startswith(f'D.{v}.L') for v in vs)] + [f'{v}.L1' for v in vs])]:
        for cov, res in [('conv', conv), ('HC3', hc3), ('HAC', hac)]:
            conj[f'conjunta_{etiqueta}_{cov}_p'] = _f(res, cols)[1] if cols else np.nan
    diag = dict(BG6_p=float(acorr_breusch_godfrey(conv, nlags=6)[1]), BG12_p=float(acorr_breusch_godfrey(conv, nlags=12)[1]),
                BG24_p=float(acorr_breusch_godfrey(conv, nlags=24)[1]),
                BP_p=float(het_breuschpagan(conv.resid, D)[1]), JB_p=float(jarque_bera(conv.resid)[1]))
    fila = dict(modelo=nombre, dependiente=y_name, criterio=criterio, p=p,
                q=', '.join(f'{c}={q[c]}' for c in xcols), n=n, parametros=k,
                inicio=str(D.index.min().date()), fin=str(D.index.max().date()),
                AIC=float(a.aic), BIC=float(a.bic), R2_niveles=float(r2_niv),
                R2adj_niveles=float(1 - (1 - r2_niv) * (n - 1) / (n - k)), R2_diferencias=r2_dif,
                R2adj_diferencias=float(conv.rsquared_adj), coef_saldo_L1=float(conv.params[f'{y_name}.L1']),
                p_en_maximo=bool(p == MAX_P), **b, **conj, **diag)
    if not completo:
        return fila, conv, D, dy, (p, q)
    # [C43] Tabla de coeficientes con EE, IC 95% y p-valores con las tres covarianzas.
    tabla = []
    for c in conv.params.index:
        f = dict(termino=c, coef=float(conv.params[c]))
        for cov, res in [('conv', conv), ('HC3', hc3), ('HAC', hac)]:
            ci = res.conf_int().loc[c]
            f.update({f'ee_{cov}': float(res.bse[c]), f'IC95_inf_{cov}': float(ci[0]),
                      f'IC95_sup_{cov}': float(ci[1]), f'p_{cov}': float(res.pvalues[c])})
        tabla.append(f)
    pd.DataFrame(tabla).to_csv(salida / f'05_coeficientes_{nombre}.csv', index=False, encoding='utf-8-sig')
    # [C44] Pruebas por producto (diferencias + nivel) con ajuste de Holm.
    prod = []
    for cov, res in ([('conv', conv), ('HC3', hc3), ('HAC', hac)] if vs else []):
        ps = [_f(res, [c for c in D if c.startswith(f'D.{v}.L')] + [f'{v}.L1'])[1] for v in vs]
        holm = multipletests(ps, method='holm')[1]
        for v, pv, ph in zip(vs, ps, holm):
            prod.append(dict(volatilidad=v, covarianza=cov, p=pv, p_Holm=ph))
    pd.DataFrame(prod).to_csv(salida / f'05_pruebas_por_producto_{nombre}.csv', index=False, encoding='utf-8-sig')
    # [C45] Multiplicadores de largo plazo (−θ/λ) por método delta: condicionados a validar la relación en niveles.
    lam = float(conv.params[f'{y_name}.L1'])
    V = conv.cov_params()
    mult = []
    for c in ['const'] + [f'{x}.L1' for x in xcols]:
        th = float(conv.params[c])
        g = pd.Series(0.0, index=conv.params.index)
        g[c], g[f'{y_name}.L1'] = -1 / lam, th / lam ** 2
        se = float(np.sqrt(g @ V @ g))
        m = -th / lam
        mult.append(dict(variable=c, multiplicador=m, ee_delta=se, p_aprox=float(2 * norm.sf(abs(m / se))),
                         IC95_inf=m - 1.96 * se, IC95_sup=m + 1.96 * se,
                         interpretacion='Solo interpretable como efecto de largo plazo si la relación en niveles queda validada'))
    pd.DataFrame(mult).to_csv(salida / f'05_multiplicadores_{nombre}.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame({nivel: v for nivel, v in cv.items()}).to_csv(salida / f'04_valores_criticos_{nombre}.csv', encoding='utf-8-sig')
    return fila, conv, D, dy, (p, q)

In [ ]:
def _supf(Z, e, Zb_fn, cand):
    base = np.sum((e - Z @ np.linalg.lstsq(Z, e, rcond=None)[0]) ** 2)
    stats = []
    for tau in cand:
        W = np.column_stack([Z, Zb_fn(tau)])
        ssr = np.sum((e - W @ np.linalg.lstsq(W, e, rcond=None)[0]) ** 2)
        m = W.shape[1] - Z.shape[1]
        stats.append(((base - ssr) / m) / (ssr / (len(e) - W.shape[1])))
    return np.array(stats)


def quiebre(conv, D, dy, salida, nombre, B=499, seed=SEMILLA):
    """Quandt–Andrews (sup-F) con 15% de recorte, p-valor por bootstrap de regresores fijos (Hansen, 2000):
    (a) cambio de nivel (solo constante) y (b) cambio de todos los coeficientes. Chow en 2020-03 y 2021-01."""
    Z, e = D.to_numpy(), dy.to_numpy()
    n = len(e)
    cand = list(range(int(.15 * n), int(.85 * n)))
    paso = lambda t: (np.arange(n) >= t).astype(float)[:, None]
    todo = lambda t: Z * paso(t)
    rng = np.random.default_rng(seed)
    resid = conv.resid.to_numpy()
    out = {}
    for etiqueta, fn in [('nivel', paso), ('todos_los_coeficientes', todo)]:
        s = _supf(Z, e, fn, cand)
        j = int(np.argmax(s))
        boot = [ _supf(Z, rng.choice(resid, n, replace=True), fn, cand).max() for _ in range(B)]
        out[etiqueta] = dict(supF=float(s[j]), fecha=str(D.index[cand[j]].date()),
                             p_bootstrap=float(np.mean(np.array(boot) >= s[j])))
    chow = []
    for fecha in ['2020-03-01', '2021-01-01']:
        t = int(np.searchsorted(D.index, pd.Timestamp(fecha)))
        for etiqueta, fn in [('nivel', paso), ('todos_los_coeficientes', todo)]:
            W = np.column_stack([Z, fn(t)])
            r = OLS(e, W).fit()
            R = np.zeros((W.shape[1] - Z.shape[1], W.shape[1]))
            R[:, Z.shape[1]:] = np.eye(W.shape[1] - Z.shape[1])
            ft = r.f_test(R)
            chow.append(dict(fecha=fecha, tipo=etiqueta, F=float(np.squeeze(ft.fvalue)), p=float(ft.pvalue)))
    filas = [dict(prueba='Quandt–Andrews sup-F', tipo=k, **v) for k, v in out.items()]
    pd.concat([pd.DataFrame(filas), pd.DataFrame(chow).assign(prueba='Chow')]).to_csv(
        salida / f'06_quiebre_{nombre}.csv', index=False, encoding='utf-8-sig')
    # CUSUM y CUSUMSQ (bandas CUSUMSQ por simulación con el mismo diseño).
    ols_obj = OLS(e, Z).fit()
    k = Z.shape[1]
    skip = next(m for m in range(k + 1, n - 12) if np.linalg.matrix_rank(Z[:m]) == k)
    rec = recursive_olsresiduals(ols_obj, skip=skip, alpha=.95)
    w = rec[3][skip:]
    cusum, bandas = rec[5][1:], rec[6]
    sq = np.cumsum(w ** 2) / np.sum(w ** 2)
    ref = np.arange(1, len(w) + 1) / len(w)
    sims = []
    for _ in range(999):
        zz = rng.standard_normal(len(w))
        ss = np.cumsum(zz ** 2) / np.sum(zz ** 2)
        sims.append(np.max(np.abs(ss - ref)))
    c0 = float(np.percentile(sims, 95))
    pd.DataFrame({'CUSUM': cusum, 'CUSUM_inf': bandas[0], 'CUSUM_sup': bandas[1], 'CUSUMSQ': sq,
                  'CUSUMSQ_inf': ref - c0, 'CUSUMSQ_sup': ref + c0}).to_csv(
        salida / f'06_cusum_cusumsq_{nombre}.csv', index=False, encoding='utf-8-sig')
    estab = dict(CUSUM_estable=bool(np.all((cusum >= bandas[0]) & (cusum <= bandas[1]))),
                 CUSUMSQ_estable=bool(np.all(np.abs(sq - ref) <= c0)))
    return out, pd.DataFrame(chow), estab

In [ ]:
# En cada extracción b: (1) se extraen los parámetros GARCH de N(θ̂, Σ̂ robusta) y se recalcula σt;
# (2) se reestima el UECM con la misma especificación y se extrae β* de N(β̂_b, V̂_b).
# Los percentiles de β* combinan la incertidumbre de ambas etapas. Se usan percentiles y no media/varianza
# porque algunas extracciones producen σt casi planas y coeficientes extremos (identificación débil).
def bootstrap_regresores(df, modelos, ad, y_name, xcols, fijo, p, q, B, seed, salida, nombre, conv):
    rng = np.random.default_rng(seed)
    draws, Fs = [], []
    for b in range(B):
        d2 = ad.copy()
        for c, r in modelos.items():
            theta, V = r.params.to_numpy(), r.param_cov.to_numpy()
            fam = 'EGARCH' if 'EGARCH' in type(r.model.volatility).__name__ else ('GJR-GARCH' if 'gamma[1]' in r.params else 'GARCH')
            dist = r.model.distribution.name
            for _ in range(100):
                th = rng.multivariate_normal(theta, V)
                pr = pd.Series(th, index=r.params.index)
                ok = pr['omega'] > 0 and _persistencia(fam, pr) < 0.999
                if fam != 'EGARCH':
                    ok = ok and pr['alpha[1]'] >= 0 and pr['beta[1]'] >= 0
                if 'nu' in pr:
                    ok = ok and pr['nu'] > (2.05 if 'Student' in dist else 1.01)
                if 'eta' in pr:
                    ok = ok and pr['eta'] > 2.05 and abs(pr['lambda']) < 1
                if ok:
                    break
            else:  # sin extracción admisible: se conserva la estimación puntual
                th = theta
            d2[f'vol_{c}'] = r.model.fix(th).conditional_volatility.reindex(d2.index)
        Dm, dy = disenio_uecm(d2[y_name], d2[xcols], p, q, fijo)
        Dm = Dm[conv.params.index]
        r2 = OLS(dy, Dm).fit()
        draws.append(rng.multivariate_normal(r2.params.to_numpy(), r2.cov_params().to_numpy()))
        Fs.append(_f(r2, [f'{y_name}.L1'] + [f'{c}.L1' for c in xcols])[0])
    Dw = np.array(draws)
    tabla = pd.DataFrame({'termino': conv.params.index, 'coef_puntual': conv.params.values,
                          'ee_convencional': conv.bse.values,
                          'mediana_bootstrap': np.median(Dw, 0),
                          'IC95_inf_bootstrap': np.percentile(Dw, 2.5, 0),
                          'IC95_sup_bootstrap': np.percentile(Dw, 97.5, 0),
                          'IC90_inf_bootstrap': np.percentile(Dw, 5, 0),
                          'IC90_sup_bootstrap': np.percentile(Dw, 95, 0)})
    signo = np.sign(conv.params.values)
    mismo = (np.sign(Dw) == signo).mean(0)
    tabla['proporcion_mismo_signo'] = mismo
    tabla['p_bootstrap_dos_colas'] = np.minimum(1, 2 * (1 - mismo))
    tabla.to_csv(salida / f'07_bootstrap_regresores_generados_{nombre}.csv', index=False, encoding='utf-8-sig')
    return tabla, np.array(Fs)

In [ ]:
def toda_yamamoto(ad, vols, ordenes, fijo, salida, y_name='balanza', hold=15):
    filas, sel = [], []
    for v in vols:
        z = ad[[y_name, v]].copy()
        z = (z - z.mean()) / z.std()
        S = fijo.loc[z.index]
        crit = []
        for k in range(1, 13):
            m = VAR(z.iloc[hold - k:], exog=S.iloc[hold - k:]).fit(k, trend='c')
            crit.append(dict(variable=v, k=k, AIC=float(m.aic), BIC=float(m.bic), n=int(m.nobs)))
        cr = pd.DataFrame(crit)
        sel.append(cr)
        k = int(cr.loc[cr.AIC.idxmin(), 'k'])
        # dmax principal: mayor orden de integración del par según el bloque 3 (mínimo 1); el otro valor es sensibilidad.
        orden = 2 if any('I(2)' in ordenes.get(x, '') for x in [y_name, v]) else 1
        for d in (1, 2):
            Dm = pd.DataFrame({'const': 1.0}, index=z.index)
            for lag in range(1, k + d + 1):
                for c in z:
                    Dm[f'{c}.L{lag}'] = z[c].shift(lag)
            Dm = Dm.join(S).iloc[hold:]
            R = np.zeros((k, Dm.shape[1]))
            for lag in range(1, k + 1):
                R[lag - 1, Dm.columns.get_loc(f'{v}.L{lag}')] = 1
            for cov in ['conv', 'HC3']:
                yy = z[y_name].iloc[hold:]
                r = OLS(yy, Dm).fit() if cov == 'conv' else OLS(yy, Dm).fit(cov_type='HC3')
                bb = R @ r.params.to_numpy()
                W = float(bb @ np.linalg.solve(R @ r.cov_params().to_numpy() @ R.T, bb))
                filas.append(dict(variable=v, k_AIC=k, k_BIC=int(cr.loc[cr.BIC.idxmin(), 'k']), dmax=d,
                                  dmax_principal=d == orden, covarianza=cov, n=int(r.nobs), MWald=W, gl=k,
                                  p=float(chi2.sf(W, k)), BG12_p=float(acorr_breusch_godfrey(OLS(yy, Dm).fit(), nlags=12)[1])))
    t = pd.DataFrame(filas)
    for (d, cov), g in t.groupby(['dmax', 'covarianza']):
        t.loc[g.index, 'p_Holm'] = multipletests(g.p, method='holm')[1]
    t.to_csv(salida / '08_toda_yamamoto.csv', index=False, encoding='utf-8-sig')
    pd.concat(sel).to_csv(salida / '08_toda_yamamoto_seleccion_k.csv', index=False, encoding='utf-8-sig')
    return t

In [ ]:
def _nota(fig, texto):
    fig.text(0.01, 0.005, f'Nota. Elaboración propia con base en datos de {texto}.', fontsize=8, color='#444444')


def figuras(df, ad, sel_vol, principal_conv, D, dy, ty, rejilla_bic, rejilla_aic, salida):
    plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.facecolor': 'white'})
    # Figura: saldo comercial mensual y anual.
    fig, ax = plt.subplots(2, 1, figsize=(11, 7))
    ax[0].plot(df.index, df.balanza, color='#1F4E79'); ax[0].axhline(0, color='k', lw=.6)
    ax[0].set_title('Saldo comercial mensual (US$ millones)')
    anual = df.balanza.groupby(df.index.year).sum()
    ax[1].bar(anual.index, anual.values, color='#B22222'); ax[1].set_title('Saldo comercial anual (US$ millones)')
    fig.tight_layout(rect=[0, .03, 1, 1]); _nota(fig, FUENTE_COMERCIO)
    fig.savefig(salida / 'fig01_saldo_comercial.png', dpi=200); plt.close(fig)
    # Figura: volatilidad condicional seleccionada y medida móvil [C31].
    fig, axs = plt.subplots(3, 2, figsize=(12, 9))
    for a, c in zip(axs.flat, PRECIOS):
        info = sel_vol.set_index('serie').loc[c]
        a.plot(df.index, df[f'vol_{c}'], label=f'{info.varianza} ({info.distribucion})', color='#1F4E79')
        a.plot(df.index, df[f'volm_{c}'], label='Desv. est. móvil 12 meses', color='#E36C09', lw=.9)
        a.set_title(NOMBRES[c]); a.set_ylabel('σ mensual (%)'); a.legend(fontsize=7)
    axs.flat[-1].axis('off')
    fig.tight_layout(rect=[0, .03, 1, 1]); _nota(fig, FUENTE_PRECIOS)
    fig.savefig(salida / 'fig02_volatilidades.png', dpi=200); plt.close(fig)
    # Figura: correlaciones entre volatilidades [C23].
    cols = [f'vol_{c}' for c in PRECIOS]
    m = ad[cols + ['balanza', 'saldo_norm']].corr()
    etiquetas = [NOMBRES[c] for c in PRECIOS] + ['Saldo (US$)', 'Saldo normalizado']
    fig, a = plt.subplots(figsize=(7.5, 6))
    im = a.imshow(m.values, cmap='RdBu_r', vmin=-1, vmax=1)
    a.set_xticks(range(len(etiquetas)), etiquetas, rotation=35, ha='right'); a.set_yticks(range(len(etiquetas)), etiquetas)
    for i in range(len(etiquetas)):
        for j in range(len(etiquetas)):
            a.text(j, i, f'{m.values[i, j]:.2f}', ha='center', va='center', fontsize=9,
                   color='white' if abs(m.values[i, j]) > .6 else 'black')
    fig.colorbar(im, fraction=.046); a.set_title('Correlaciones entre volatilidades condicionales y saldo')
    fig.tight_layout(rect=[0, .04, 1, 1]); _nota(fig, f'{FUENTE_PRECIOS}; {FUENTE_COMERCIO}')
    fig.savefig(salida / 'fig03_correlaciones_volatilidades.png', dpi=200); plt.close(fig)
    m.to_csv(salida / 'fig03_correlaciones_volatilidades.csv', encoding='utf-8-sig')
    # Figura: AIC y BIC por número de rezagos del saldo [C38].
    fig, a = plt.subplots(figsize=(9, 4.5))
    for rej, crit, color in [(rejilla_aic, 'AIC', '#1F4E79'), (rejilla_bic, 'BIC', '#E36C09')]:
        mejor = rej.groupby('p')[crit].min()
        a.plot(mejor.index, mejor.values, marker='o', label=crit, color=color)
    a.set_xlabel('Rezagos del saldo (p)'); a.set_ylabel('Criterio de información'); a.legend()
    a.set_title('Selección de rezagos con dummies estacionales (mínimo sobre q)')
    fig.tight_layout(rect=[0, .04, 1, 1]); _nota(fig, f'{FUENTE_COMERCIO}; {FUENTE_PRECIOS}; SECMCA')
    fig.savefig(salida / 'fig04_seleccion_rezagos.png', dpi=200); plt.close(fig)
    # Figura: CUSUM y CUSUMSQ del modelo principal [C36].
    cs = pd.read_csv(salida / '06_cusum_cusumsq_principal.csv')
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(cs.CUSUM); axs[0].plot(cs.CUSUM_inf, 'r--'); axs[0].plot(cs.CUSUM_sup, 'r--'); axs[0].set_title('CUSUM (bandas 5%)')
    axs[1].plot(cs.CUSUMSQ); axs[1].plot(cs.CUSUMSQ_inf, 'r--'); axs[1].plot(cs.CUSUMSQ_sup, 'r--'); axs[1].set_title('CUSUMSQ (bandas 5%)')
    fig.tight_layout(rect=[0, .05, 1, 1]); _nota(fig, f'{FUENTE_COMERCIO}; {FUENTE_PRECIOS}; SECMCA')
    fig.savefig(salida / 'fig05_estabilidad.png', dpi=200); plt.close(fig)
    # Figura: Toda–Yamamoto, p-valores de Holm convencionales y HC3 [C46].
    t = ty[ty.dmax_principal]
    fig, a = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(PRECIOS))
    for j, cov in enumerate(['conv', 'HC3']):
        g = t[t.covarianza == cov].set_index('variable').reindex([f'vol_{c}' for c in PRECIOS])
        barras = a.bar(x + (j - .5) * .35, g.p_Holm, .35, label='Convencional' if cov == 'conv' else 'HC3')
        for bar, v in zip(barras, g.p_Holm):
            a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + .01, '< 0.001' if v < .001 else f'{v:.3f}',
                   ha='center', fontsize=8)
    a.axhline(.05, color='r', ls='--', lw=.8); a.set_xticks(x, [NOMBRES[c] for c in PRECIOS])
    a.set_ylabel('p-valor ajustado por Holm'); a.set_ylim(0, 1.1); a.legend()
    a.set_title('Precedencia predictiva (Toda–Yamamoto), dmax principal')
    fig.tight_layout(rect=[0, .04, 1, 1]); _nota(fig, f'{FUENTE_COMERCIO}; {FUENTE_PRECIOS}')
    fig.savefig(salida / 'fig06_toda_yamamoto.png', dpi=200); plt.close(fig)

In [ ]:
SALIDAS = [
    ('00_diagnostico_precios_repetidos.csv / 00_diagnostico_retornos_extremos.csv', 'C25', 'Valores repetidos y retornos extremos para verificar contra la fuente'),
    ('00_diccionario_variables.csv', 'C27', 'Diccionario de variables, fuentes, unidades y transformaciones'),
    ('01_volatilidad_candidatos.csv', 'C28, C29, C30', 'Todos los candidatos GARCH/GJR/EGARCH × distribución × AR(0–3) con diagnósticos'),
    ('01_volatilidad_seleccion_y_diagnosticos.csv', 'C28, C29, C30, C32', 'Modelo elegido, LR contra varianza constante, ARCH-LM, Ljung–Box, persistencia'),
    ('01_volatilidad_parametros.csv', 'C28, C34', 'Parámetros con errores estándar robustos'),
    ('01_anexo_10_2_salidas_GARCH.txt', 'C34', 'Salidas completas de cada modelo de volatilidad (anexo 10.2)'),
    ('02_integracion_resumen.csv / 02_integracion_detalle.csv', 'C35, C36', 'ADF, PP, KPSS y Zivot–Andrews con c y ct; orden de integración'),
    ('03_rezagos_*.csv, fig04_seleccion_rezagos.png', 'C38', 'Rejilla de rezagos con AIC y BIC'),
    ('04_comparacion_modelos.csv', 'C19, C30, C31, C36, C38–C41, C44', 'Todas las especificaciones: Bounds F, t y Fx, diagnósticos y pruebas conjuntas'),
    ('04_valores_criticos_*.csv', 'C40', 'Valores críticos de muestra finita simulados para F, t y Fx'),
    ('05_coeficientes_*.csv', 'C43', 'Coeficientes con EE, IC 95% y p-valores: convencional, HC3 y HAC'),
    ('05_pruebas_por_producto_*.csv', 'C44', 'Prueba por volatilidad con ajuste de Holm'),
    ('05_multiplicadores_*.csv', 'C45', 'Multiplicadores de largo plazo por método delta (condicionados)'),
    ('05_VIF.csv / 05_correlacion_controles.csv', 'C42', 'Colinealidad entre TCN, ITCER, remesas y volatilidades'),
    ('06_quiebre_principal.csv / 06_cusum_cusumsq_principal.csv / fig05_estabilidad.png', 'C36', 'Quandt–Andrews, Chow 2020–2021, CUSUM y CUSUMSQ'),
    ('07_bootstrap_regresores_generados_principal.csv', 'C33', 'Errores estándar corregidos por la estimación de la primera etapa'),
    ('08_toda_yamamoto.csv / fig06_toda_yamamoto.png', 'C46', 'Precedencia predictiva con HC3, Holm, dummies y orden del VAR'),
    ('fig02_volatilidades.png', 'C31', 'Volatilidad seleccionada frente a desviación estándar móvil'),
    ('fig03_correlaciones_volatilidades.png', 'C23', 'Correlaciones entre volatilidades y con el saldo'),
    ('fig01 a fig06', 'C21', 'Notas de fuente con la institución de origen'),
    ('09_datos_transformados.csv', 'C27', 'Retornos, volatilidades y series usadas en las estimaciones'),
]


def readme(salida, y):
    lineas = ['# Resultados del código corregido v1', '',
              f'Variable dependiente principal: {NOMBRES.get(y, y)}.', '',
              '| Archivo | Ítem del checklist | Contenido |', '| --- | --- | --- |']
    lineas += [f'| {a} | {b} | {c} |' for a, b, c in SALIDAS]
    (salida / 'README_resultados.md').write_text('\n'.join(lineas) + '\n', encoding='utf-8')

In [ ]:
def main(argv=None):
    ap = argparse.ArgumentParser(description='Código corregido v1 de la tesis')
    ap.add_argument('--archivo', default=ARCHIVO)
    ap.add_argument('--salida', default=SALIDA)
    ap.add_argument('--bootstrap', type=int, default=300)
    ap.add_argument('--nsim', type=int, default=5000)
    ap.add_argument('--dependiente', choices=['saldo_norm', 'balanza'], default='saldo_norm',
                    help='saldo_norm: saldo/(X+M)×100 [C19]; balanza: saldo en US$ millones')
    if argv is None and 'ipykernel' in sys.modules:
        argv = []
    args = ap.parse_args(argv)
    archivo, salida = Path(args.archivo), Path(args.salida)
    salida.mkdir(parents=True, exist_ok=True)
    df, bandera = cargar_datos(archivo)
    diccionario(archivo, salida)
    rep = diagnostico_precios(df, salida)
    for c in PRECIOS:
        df[f'r_{c}'] = 100 * np.log(df[c]).diff()
    sel_vol, modelos = volatilidades(df, salida)
    vols = [f'vol_{c}' for c in PRECIOS]
    xcols = vols + CONTROLES
    variables_ur = ['balanza', 'saldo_norm'] + vols + CONTROLES + [f'volg_{c}' for c in PRECIOS] + [f'volm_{c}' for c in PRECIOS]
    ur = integracion(df, variables_ur, salida)
    ordenes = dict(zip(ur.variable, ur.orden))
    # Muestra ARDL: meses con todas las volatilidades seleccionadas disponibles (sin comprimir meses).
    ad = df[['balanza', 'saldo_norm', 'covid'] + xcols + [f'volg_{c}' for c in PRECIOS]].dropna().copy()
    ad.index = pd.DatetimeIndex(ad.index, freq='MS')
    S = dummies_estacionales(ad.index)
    # VIF y correlaciones de los controles [C42].
    vif = pd.DataFrame({'variable': xcols, 'VIF': [variance_inflation_factor(add_constant(ad[xcols]).to_numpy(), i + 1)
                                                    for i in range(len(xcols))]})
    vif.to_csv(salida / '05_VIF.csv', index=False, encoding='utf-8-sig')
    ad[CONTROLES].corr().to_csv(salida / '05_correlacion_controles.csv', encoding='utf-8-sig')
    filas = []
    # Modelo principal: dummies estacionales, BIC [C38, C39].
    y = args.dependiente
    otra = 'balanza' if y == 'saldo_norm' else 'saldo_norm'
    p, q, rej_bic = seleccionar_rezagos(ad[y], ad[xcols], S, 'BIC', 'principal', salida)
    fila, conv, D, dy, _ = estimar(ad, y, xcols, S, 'BIC', 'principal', salida, vols,
                                   nsim=args.nsim, p=p, q=q, completo=True)
    filas.append(fila)
    # Robustez: AIC con dummies [C38].
    fila_aic, *_ = estimar(ad, y, xcols, S, 'AIC', 'aic', salida, vols, nsim=2000, completo=True)
    filas.append(fila_aic)
    rej_aic = pd.read_csv(salida / '03_rezagos_aic_AIC.csv')
    # Réplica de la especificación de la tesis (sin dummies, q = 1, p por AIC), para comparar.
    rep_rows = [(_ardl(ad.balanza, ad[xcols], pp, {c: 1 for c in xcols}, None), pp) for pp in range(1, MAX_P + 1)]
    p_orig = min(rep_rows, key=lambda r: r[0].aic)[1]
    fila_orig, *_ = estimar(ad, 'balanza', xcols, None, 'AIC', 'replica_especificacion_tesis', salida, vols, nsim=2000,
                            p=p_orig, q={c: 1 for c in xcols})
    filas.append(fila_orig)
    # Quiebre estructural [C36].
    qb, chow, estab = quiebre(conv, D, dy, salida, 'principal')
    filas[0].update(estab)
    filas[0].update({f'quiebre_{k}_{m}': v[m] for k, v in qb.items() for m in ('supF', 'fecha', 'p_bootstrap')})
    S_cov = S.join(ad[['covid']])
    fila_cov, *_ = estimar(ad, y, xcols, S_cov, 'BIC', 'covid', salida, vols, p=p, q=q)
    filas.append(fila_cov)
    fecha_q = pd.Timestamp(qb['nivel']['fecha'])
    S_q = S.assign(quiebre=(ad.index >= fecha_q).astype(float))
    fila_q, *_ = estimar(ad, y, xcols, S_q, 'BIC', 'quiebre_nivel', salida, vols, p=p, q=q)
    filas.append(fila_q)
    # La otra medida del saldo (US$ millones o normalizado) [C19].
    fila_n, *_ = estimar(ad, otra, xcols, S, 'BIC', f'dependiente_{otra}', salida, vols, completo=True)
    filas.append(fila_n)
    # Volatilidad GARCH(1,1) uniforme [C30] y desviación estándar móvil [C31].
    for pref, nombre, base in [('volg', 'vol_garch11_uniforme', ad),
                               ('volm', 'vol_desv_movil_12m', df[[y] + [f'volm_{c}' for c in PRECIOS] + CONTROLES].dropna())]:
        d2 = base.copy()
        d2.index = pd.DatetimeIndex(d2.index, freq='MS')
        for c in PRECIOS:
            d2[f'vol_{c}'] = d2[f'{pref}_{c}']
        f2, *_ = estimar(d2, y, xcols, dummies_estacionales(d2.index), 'BIC', nombre, salida, vols)
        filas.append(f2)
    # Exclusión de un regresor a la vez: dependencia del F respecto de cada variable [C41, C42].
    for c in xcols:
        cols = [x for x in xcols if x != c]
        f2, *_ = estimar(ad, y, cols, S, 'BIC', f'excluir_{c}', salida, vols,
                         p=p, q={x: q[x] for x in cols})
        filas.append(f2)
    comp = pd.DataFrame(filas)
    comp.to_csv(salida / '04_comparacion_modelos.csv', index=False, encoding='utf-8-sig')
    # Regresores generados [C33].
    boot, Fboot = bootstrap_regresores(df, modelos, ad, y, xcols, S, p, q, args.bootstrap, SEMILLA, salida, 'principal', conv)
    # Toda–Yamamoto [C46].
    ty = toda_yamamoto(ad, vols, ordenes, S, salida, y_name=y)
    figuras(df, ad, sel_vol, conv, D, dy, ty, rej_bic, rej_aic, salida)
    df.to_csv(salida / '09_datos_transformados.csv', index_label='fecha', encoding='utf-8-sig')
    readme(salida, y)
    principal = comp.iloc[0].to_dict()
    resumen = dict(archivo=str(archivo), sha256=hashlib.sha256(archivo.read_bytes()).hexdigest(),
                   versiones={'arch': arch.__version__, 'statsmodels': statsmodels.__version__,
                              'pandas': pd.__version__, 'numpy': np.__version__},
                   cardamomo_meses_interpolados=int(bandera.sum()),
                   especificacion_principal=dict(dependiente=y, p=p, q=q, dummies_estacionales=True, criterio='BIC'),
                   muestra=dict(inicio=principal['inicio'], fin=principal['fin'], n=principal['n']),
                   volatilidad=sel_vol.to_dict(orient='records'),
                   integracion=ur[['variable', 'orden']].to_dict(orient='records'),
                   principal=principal, quiebre=qb,
                   bootstrap=dict(B=args.bootstrap, F_mediana=float(np.median(Fboot)),
                                  F_p5=float(np.percentile(Fboot, 5)), F_p95=float(np.percentile(Fboot, 95))))
    (salida / 'resumen_v1.json').write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return dict(df=df, ad=ad, comparacion=comp, volatilidad=sel_vol, integracion=ur, toda_yamamoto=ty,
                bootstrap=boot, resumen=resumen, salida=salida)


if __name__ == '__main__':
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = main()
    c = res['comparacion']
    print(c[['modelo', 'criterio', 'p', 'n', 'AIC', 'BIC', 'F', 'F_decision', 't_decision', 'Fx_decision']].to_string(index=False))
    print('\nResultados en', res['salida'].resolve())